# Comparison with published essential-gene sets

This notebook compares significant genes from the CRISPRi screen across time points and summary strategies, and relates depleted-gene calls to essential genes reported by Lee et al. (2015).

In [74]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.ticker import MultipleLocator, NullLocator
from matplotlib_venn import venn3, venn2

## 1. Prepare significant-gene sets

Result tables are restricted to PA locus tags represented by at least three sgRNAs, with one row retained per gene. Depending on the requested set type, the helper functions return depleted genes, enriched genes, or their union (`both`). The significance labels themselves were assigned in the preceding screen-analysis workflow.

In [75]:
# Function to get the lists. Not used here 

def list_enriched_depleted_both(filepath):

    p = Path(filepath)
    filename = p.stem
    fileparent = p.parent
    destination_folder = (fileparent / "lists_for_venn").mkdir(exist_ok=True)
    destination_filepath = destination_folder / filename 

    df = pd.read_csv(filepath, low_memory=False)

    # Use only genes that have at least 3 sgRNA
    df = df[df['nb_of_sgrna'] >= 3]
  
    # Take only PA genes (start with PA)
    df = df[(df['Gene'].str.startswith('PA'))]
    
    # Keep only one row per gene 
    df = df.drop_duplicates(subset='Gene')
   
    # Drop sgrna columns 
    df = df.drop(columns=['sgrna', 'LFC_individual'])

    # Get the enriched and depleted lists 
    df_enriched = df[df['hue'] == 'enriched']
    df_depleted = df[df['hue'] == 'depleted']
    df_both = df[(df['hue'] == 'enriched') | (df['hue'] == 'depleted')]

    # Save the lists
    destination_filepath_enriched = f"{destination_filepath}_enriched.csv"
    df_enriched.to_csv(destination_filepath_enriched, index=False)

    destination_filepath_depleted = f"{destination_filepath}_depleted.csv"
    df_depleted.to_csv(destination_filepath_depleted, index=False)

    destination_filepath_both = f"{destination_filepath}_both.csv"
    df_both.to_csv(destination_filepath_both, index=False)

## 2. Load the Lee et al. reference sets

The supplementary workbook provides all reported essential genes and a more restrictive general-essential set. For the LB-specific comparison in this notebook, a gene is included when it was classified as essential in at least one of the six LB datasets (`n = 1`). 

In [ ]:
# Make the list of essential genes from Lee

filepath = "path/to/xlsx_file_from_lee_paper"

df_lee_all = pd.read_excel(filepath, sheet_name="All essential genes", header=1)
df_lee_general = pd.read_excel(filepath, sheet_name="General essential genes", header=1)

# All genes in the lists 
set_lee_all = set(df_lee_all['PA number'])
set_lee_general = set(df_lee_general['PA number'])

# Essential genes only in LB (in at least n out of 6)
n = 1
lee_lb = df_lee_all[(df_lee_all['LB sum (of 6)'] >= n)]
set_lee_lb = set(lee_lb['PA number'])


In [77]:
def make_set(filepath, set_type):

    df = pd.read_csv(filepath, low_memory=False)

    # Use only genes that have at least 3 sgRNA
    df = df[df['nb_of_sgrna'] >= 3]
  
    # Take only PA genes (start with PA)
    df = df[(df['Gene'].str.startswith('PA'))]
    
    # Keep only one row per gene 
    df = df.drop_duplicates(subset='Gene')
   
    # Drop sgrna columns 
    df = df.drop(columns=['sgrna', 'LFC_individual'])

    if set_type == "depleted":
        # Get the depleted lists 
        df_depleted = df[df['hue'] == 'depleted']
        return set(df_depleted['Gene'])
    
    elif set_type == "enriched":
        # Get the enriched lists 
        df_enriched = df[df['hue'] == 'enriched']
        return set(df_enriched['Gene'])
    
    elif set_type == "both":
        # Get the enriched and depeletd lists 
        df_both = df[(df['hue'] == 'enriched') | (df['hue'] == 'depleted')]
        return set(df_both['Gene'])
    
    else:
        print("Indicate the type of list you want: depleted, enriched or both.")

## 3. Load median and second-best result tables

The input directory contains results summarized using two strategies. Filenames without the `_2` suffix represent the median-based results; `_2` denotes the second-best-guide results. Sets are generated for each medium and time point so the strategies can be compared consistently.

In [ ]:
data_folder = Path("path/to/Median and 2nd Best_data folder")
files =  {f.stem.split(' ')[2]: f for f in data_folder.iterdir() if f.is_file()}
files

In [79]:
list_both = {key: make_set(value, 'both') for key, value in files.items()}

In [80]:
list_depleted = {key: make_set(value, 'depleted') for key, value in files.items()}

## 4. Compare time points, summary strategies, and reference genes

The Venn-diagram grid compares 24-hour with 48-hour calls, median with second-best summaries, and selected LB depleted-gene sets with the Lee LB reference. For the time-point and method comparisons, `both` contains any gene labelled enriched or depleted; the Lee comparisons use depleted genes only.

In [ ]:
fig, axs = plt.subplots(nrows=3, ncols=6, figsize=(16,8))
set_colors = ('red', 'blue')
# Plot the 24h-48h comparisons 
set_labels = ['24h', '48h']
venn2([list_both['24hLB'], list_both['48hLB']], set_labels=set_labels, ax=axs[0,0], set_colors=set_colors)
venn2([list_both['24hGlc'], list_both['48hGlc']], set_labels=set_labels, ax=axs[0,1], set_colors=set_colors)
venn2([list_both['24hSucc'], list_both['48hSucc']], set_labels=set_labels, ax=axs[0,2], set_colors=set_colors)
venn2([list_both['24hLB_2'], list_both['48hLB_2']], set_labels=set_labels, ax=axs[0,3], set_colors=set_colors)
venn2([list_both['24hGlc_2'], list_both['48hGlc_2']], set_labels=set_labels, ax=axs[0,4], set_colors=set_colors)
venn2([list_both['24hSucc_2'], list_both['48hSucc_2']], set_labels=set_labels, ax=axs[0,5], set_colors=set_colors)

# Plot median vs second best
set_labels = ['median', 'second_best']
venn2([list_both['24hLB'], list_both['24hLB_2']], set_labels=set_labels, ax=axs[1,0], set_colors=set_colors)
venn2([list_both['24hGlc'], list_both['24hGlc_2']], set_labels=set_labels, ax=axs[1,1], set_colors=set_colors)
venn2([list_both['24hSucc'], list_both['24hSucc_2']], set_labels=set_labels, ax=axs[1,2], set_colors=set_colors)
venn2([list_both['48hLB'], list_both['48hLB_2']], set_labels=set_labels, ax=axs[1,3], set_colors=set_colors)
venn2([list_both['48hGlc'], list_both['48hGlc_2']], set_labels=set_labels, ax=axs[1,4], set_colors=set_colors)
venn2([list_both['48hSucc'], list_both['48hSucc_2']], set_labels=set_labels, ax=axs[1,5], set_colors=set_colors)

# Plot median 48 vs second best 24
set_labels = ['median_48', 'second_best_24']
venn2([list_both['48hLB'], list_both['24hLB_2']], set_labels=set_labels, ax=axs[2,0], set_colors=set_colors)
venn2([list_both['48hGlc'], list_both['24hGlc_2']], set_labels=set_labels, ax=axs[2,1], set_colors=set_colors)
venn2([list_both['48hSucc'], list_both['24hSucc_2']], set_labels=set_labels, ax=axs[2,2], set_colors=set_colors)

# Plot lee_LB vs the others
lee = 'Lee_LB'
venn2([list_depleted['48hLB'], set_lee_lb], set_labels=['median_48', lee], ax=axs[2,3], set_colors=set_colors)
venn2([list_depleted['24hLB_2'], set_lee_lb], set_labels=['second_best_24', lee], ax=axs[2,4], set_colors=set_colors)
venn2([list_depleted['48hLB_2'], set_lee_lb], set_labels=['second_best_48', lee], ax=axs[2,5], set_colors=set_colors)



Unless otherwise indicated, the sets include both enriched and depleted genes.

**First row:** From left to right, the first three panels compare the 24 h and 48 h sets for LB, glucose, and succinate using the median sgRNA summary method. The final three panels show the corresponding comparisons using the second-best sgRNA method.

**Second row:** From left to right, the panels compare the median and second-best sgRNA methods for LB, glucose, and succinate at 24 h, followed by the same three comparisons at 48 h.

**Third row:** The first three panels compare the 48 h median set with the 24 h second-best set for LB, glucose, and succinate. The final three panels compare LB depleted genes (using the 48 h median, 24 h second-best, and 48 h second-best methods, respectively) with the Lee et al. LB essential-gene reference set.

![Venn Diagram](figures/venn.png)

## 5. Export annotated overlap lists

The final functions divide any pair of gene sets into their intersection and the genes unique to each set. These locus tags are joined to the PAO1 genome annotation, and the resulting gene names and product descriptions are exported under `venn_lists/`.

In [ ]:
GENOME = pd.read_csv("path/to/genome_file")

In [83]:
def add_gene_fct(myset, genome=GENOME):
    df = pd.DataFrame({'Gene': list(myset)})
    result = df.merge(genome, left_on='Gene', right_on='Locus Tag', how='left')
    result = result[['Gene', 'Gene Name', 'Product Name']]
    return result

In [84]:
def make_files_with_lists(dico, cond1, cond2):

    set1 = dico[cond1]
    set2 = dico[cond2]
    comparison = f"{cond1}_vs_{cond2}"

    intersection = set1 & set2
    set1_only= set1 - set2
    set2_only = set2 - set1

    # The files will be saved in the current working directory
    output_folder = Path("venn_lists") / comparison
    output_folder.mkdir(parents=True, exist_ok=True)

    file_intersection = output_folder / f"intersection_{cond1}_vs_{cond2}.csv"
    file_set1_only = output_folder / f"{cond1}_only.csv"
    file_set2_only = output_folder / f"{cond2}_only.csv"

    df_intersection = (add_gene_fct(intersection)).sort_values(by='Gene')
    df_set1 = (add_gene_fct(set1_only)).sort_values(by='Gene')
    df_set2 = (add_gene_fct(set2_only)).sort_values(by='Gene')

    df_intersection.to_csv(file_intersection, index=False)
    df_set1.to_csv(file_set1_only, index=False)
    df_set2.to_csv(file_set2_only, index=False)

In [ ]:
list_both.keys()

In [86]:
# How to generate the lists of genes
make_files_with_lists(list_both, '24hSucc', '48hSucc')

In [87]:
# For comparison with lee
list_depleted['lee_lb'] = set_lee_lb
make_files_with_lists(list_depleted, '48hLB_2', 'lee_lb')